# 13. 短期預報上線：STEP 與作業化系統

前面三章講的是模型；這一章講模型**上線**之後的事。一個統計
模型要變成一個 24 小時運轉、對社會發布的作業化系統，會冒出
一連串不再是統計的問題：參數哪來？資料不足時怎麼辦？機率
算出來之後，要用什麼話說給誰聽？

我們用這條路上最經典的模型當主角：**STEP**（Short-Term
Earthquake Probability，Gerstenberger et al. 2005, *Nature*）
——第一個把「明天的地震」變成即時網頁地圖的系統。

## 13.1 從一條公式到一張地圖

故事從一條非常簡單的公式開始。Reasenberg & Jones (1989) 把
GR 律與 Omori 律相乘，得到主震（規模 $M_m$）之後 $t$ 天、
規模 $M$ 以上的餘震發生率：

$$R(t, M) = 10^{\,a' + b\,(M_m - M)}\,(t + c)^{-p}$$

用 62 個加州序列擬合出一組**通用參數**（$a'=-1.67$、$b=0.91$、
$p=1.08$、$c=0.05$），再積分成時窗機率 $P = 1 - e^{-\int R\,dt}$,
就能在任何主震後立刻回答「接下來一週再來一個更大的機率」。
1989 年起，這條公式就是加州餘震公告的引擎。它便宜、解析、
即時——但也有兩個公認的限制：參數是通用的，**不針對眼前
這個序列調校**；而且它只給機率<strong>，不含餘震會發生在哪裡的
資訊</strong>。

STEP 補的正是這兩個洞。它把預報鋪上空間網格，並用**三個
複雜度遞增的模型元素**組合：

1. **通用層**：序列剛開始、資料不足時，直接套用區域通用參數；
2. **序列特定層**：當序列累積了**至少 100 個**完整規模以上的
   餘震，就改用該序列自己擬合的參數；
3. **空間變化層**：對特別多產的序列，進一步允許參數在序列的
   不同子區各自擬合。

三層依各自對資料的擬合程度（AIC）自動加權——資料少時通用層
主導，資料累積後序列自己的行為接手。時變項算完後再疊上一個
**時間獨立的背景項**（來自長期危害模型），輸出從「餘震數量
機率」升級成「**未來 24 小時內任何地點發生強烈震動的機率
地圖**」。用一個合成主震畫出這個概念：

In [ ]:
from gdms_toolkit.viz import setup_plotly
setup_plotly()

In [ ]:
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from scipy.integrate import quad

from gdms_toolkit.viz import ACCENT, QUAKE_COLOR, apply_layout

# 合成情境：均勻背景 + 一個 M6.5 主震的 Omori × 空間核
x = np.linspace(0, 100, 120)
y = np.linspace(0, 100, 120)
XX, YY = np.meshgrid(x, y)
r2 = (XX - 50) ** 2 + (YY - 55) ** 2
bg = 2e-4                                        # 背景（每格每日率）
spatial = np.exp(-r2 / (2 * 12 ** 2))            # 主震的空間核（σ=12 km）

def omori_rate(t, K=8.0, c=0.05, p=1.08):
    return K / (t + c) ** p

fig = make_subplots(rows=1, cols=2, shared_yaxes=True,
                    subplot_titles=("主震後第 1 天", "主震後第 30 天"),
                    horizontal_spacing=0.03)
zmax = np.log10(bg + omori_rate(1.0) * spatial * 1e-2).max()
for col, day in [(1, 1.0), (2, 30.0)]:
    rate = bg + omori_rate(day) * spatial * 1e-2
    fig.add_trace(go.Heatmap(x=x, y=y, z=np.log10(rate), colorscale="Blues",
                             zmin=np.log10(bg) - 0.2, zmax=zmax,
                             showscale=(col == 2),
                             colorbar=dict(title="log₁₀ 率")), row=1, col=col)
    fig.add_trace(go.Scatter(x=[50], y=[55], mode="markers", showlegend=False,
                             marker=dict(symbol="star", size=13,
                                         color=QUAKE_COLOR)), row=1, col=col)
fig.update_yaxes(scaleanchor="x", row=1, col=1)
apply_layout(fig, title="STEP 概念：時變叢集項疊在時間獨立背景上，並隨時間衰減",
             height=440, hovermode="closest")
fig

第 1 天，主震周圍的發生率比背景高出兩個數量級；第 30 天，
熱區已按 Omori 律消退大半，但仍未回到背景——最終整張地圖會
退回長期危害模型。這種「時變項終將融回時不變項」的設計，讓
短期預報與國家地震危害模型天然一致，也是 STEP 系架構到今天
仍被沿用的原因。

把時間軸拉出來看更清楚。用 Reasenberg–Jones 公式與加州通用
參數，算「M6.0 主震後，未來 7 天內發生 M≥5 餘震的機率」隨
時間的變化：

In [ ]:
def p_week(t0, mm=6.0, m=5.0, a=-1.67, b=0.91, c=0.05, p=1.08):
    rate = lambda t: 10 ** (a + b * (mm - m)) * (t + c) ** -p
    integral, _ = quad(rate, t0, t0 + 7)
    return 1 - np.exp(-integral)

t0s = np.logspace(-2, 2.5, 60)
fig = go.Figure(go.Scatter(x=t0s, y=[p_week(t) * 100 for t in t0s],
                           mode="lines", name="M6.0 主震後（R–J 加州通用參數）",
                           line=dict(color=ACCENT, width=2.5)))
fig.add_hline(y=0.1, line_dash="dash", line_color="#1baf7a",
              annotation_text="平時背景（示意）")
apply_layout(fig, title="「未來 7 天內 M≥5」的機率如何隨主震後時間衰減",
             xaxis_title="主震後時間（天）", yaxis_title="7 天內機率（%）",
             xaxis_type="log", yaxis_type="log", hovermode="x")
fig

主震剛過的頭一天，一週內再來一個 M≥5 的機率有五、六成；一週
後掉到百分之十上下；一個月後只剩百分之幾；幾個月後就貼回背景
水準。**機率增益大（相對背景高
數十倍），但絕對機率從來不高**——這是短期預報的宿命，也是
後面溝通問題的根源。

順帶補上兩個「一行算式」的教訓。同樣的公式，加州參數算出
「一週內被同規模或更大地震跟隨」的機率是 10.5%，而南加州
實際觀測的前震比例只有約 6%——**模型與觀測可以差近一倍**；
換用日本的參數（Utsu 1969）算出來則是 4.2%——**同一個模型、
不同地區的參數，答案差一倍以上**。參數在地化不是講究，是
必要。

## 13.2 從模型到系統：三條路線

STEP 於 2005 年在加州上線（原網站已走入歷史；美國現行系統
是新一代的作業化餘震預報系統（Operational Aftershock
Forecasting, OAF），用 Reasenberg–Jones 或 ETAS 引擎——把 STEP
講成「USGS 現在在跑的系統」是常見的錯誤）。二十年下來，
世界上形成了三條代表性的作業化路線，第 9 章看過概要，這裡
補上工程與制度的細節：

| | 義大利 INGV | 紐西蘭 GNS/GeoNet | 美國 USGS |
|---|---|---|---|
| 引擎 | ETAS／ETES／STEP 集成，權重每週依表現更新 | 短期 STEP/ETAS＋中期 EEPAS＋長期 PPE，取最大值混成 | R–J 貝氏（境內自動）；ETAS＋萬次模擬（境外、人工啟動） |
| 節奏 | 每日午夜＋每個 M≥3.5 後，發布一週預報 | 無固定排程，隨序列演化 | 大震後約 20 分鐘自動首報 |
| 公開 | 不對大眾（僅民防體系） | 完全公開 | 完全公開 |

（三國定位的制度面對照見{doc}`第 9 章 <09_forecasting_intro>`。）
三個系統各有一個值得單獨記住的細節：

- **美國的貝氏設計**是「貝氏更新」最好的具體教材：t=0 時
  沒有任何本序列資料，預報採用涵蓋歷史序列行為範圍的通用
  先驗，因此**最寬**；隨資料累積，後驗逐步收斂到這個序列
  自己的行為。預報的不確定性帶隨時間變窄，本身就是資訊。
- **紐西蘭的混成**用「取最大值」而非相加來合成不同時間尺度
  的模型——因為短、中、長期模型描述的是**同一個**危害的
  不同視角，相加會重複計算。這個混成模型不只發預報：基督城
  重建的 50 年時變危害模型、Kaikōura 之後中紐西蘭無筋磚造
  建築的**強制補強政策**，都直接建立在它上面——短期預報
  接上長期危害與工程決策，就是第 16 章要講的時變危害。
- **義大利的把關**：候選模型要加入官方集成，唯一的強制條件
  是**必須提交過 CSEP 實驗**接受第三方檢驗。「開發者自己
  相信」不算數，這裡變成了制度。

台灣呢？第 11 章看過：2025 年大埔地震的 ETAS 快報已經具備
OAF 的全部零件——預訓練參數、每輪 7 分鐘的即時更新、震度
機率輸出。台灣缺的不是技術，是第 9 章說的那個制度問題：
權威性由誰授予、要不要對大眾發布。

上線之後，模型也會用各種方式壞掉。紐西蘭 2016 Kaikōura M7.8
是最好的實戰教案：序列初期完整規模的估計出了問題，系統只能
退回通用參數，結果**高估**了實際的餘震數。事後看，錯誤的
源頭不在模型公式，而在第 10 章講的目錄即時品質——作業化
系統的風險清單上，資料永遠排在模型前面。

## 13.3 機率很小的時候怎麼說話

2009 年 4 月 6 日，義大利 L'Aquila 發生 $M_w$ 6.3 地震，
約三百人罹難。震前該區有持續數月的群震，官方留給公眾的
訊息卻被廣泛理解為「不會有大地震」。後續的審判讓全世界的
地震學家記住了一課——但這一課常常被記成「說了會被告」，
而真相是雙面的：科學家與官員被批評的，恰恰是**沒有把
「機率確實升高了」講得夠清楚、夠強**。沉默的代價與說錯話
的代價同樣真實。這場災難直接催生了 ICEF 與「作業化地震
預報」這個領域（Jordan et al. 2011）：與其臨場即興，不如
建立常態化、權威、透明的機率發布制度。

但機率怎麼說，本身就是一門有實驗、有數據的科學，不是文案
品味。幾個已被研究確立的操作要點：

- **時窗用語**：說「在未來一週**之內**（within）」而不是
  「在下一週（in）」——後者會讓人以為事件將發生在時窗末端；
- **換一個參照系**：實驗證實有效的框架是「想像 100,000 個
  機率與此地完全相同的地方，本週我們預期其中 y 個會發生
  ……」——把抽象機率變成頻率；
- **報數量範圍比報機率範圍抗扭曲**：媒體轉述機率區間時幾乎
  只報最悲觀的上限，但轉述「預期餘震數量 0–2 個」這類範圍
  卻相對忠實；
- **機率永遠配上文字判讀**：紐西蘭 GeoNet 的預報表每一格
  都同時給平均數、範圍、機率與「unlikely／extremely
  unlikely」等判讀語彙。

量級感也要一起交給讀者。Kaikōura M7.8 之後兩週，GeoNet 的
未來 7 天預報：M5.0–5.9 至少一次 **98%**、M6.0–6.9 **41%**、
M≥7 **5%**。隔年年初它進一步發布**三情境敘事**：約 70%——
餘震照常衰減；約 25%——發生 M7.0–7.8（含隱沒帶參與和局部海嘯的可能）；
約 5%——出現比主震更大的地震（含 M8 以上板塊介面破裂）。每個情境同時給
文字、機率與潛在衝擊，並明講「機率雖低，但比主震前高了
許多」。這份文件被廣泛視為機率溝通的典範。

最後一個誠實的難題：溝通研究在 Canterbury 序列六年後的工作坊中發現，理解餘震序列行為的居民在第六年的 M5.7 來襲時比較
不驚訝——**但也因此沒有採取行動**。降低驚訝與促成行動，
可能是互相衝突的目標；而人們真正想要的是「可以行動的
資訊」，不是機率本身。這句話值得整個第二部反覆回味：機率
是我們花了四章學會計算的東西，但它只是中間產物。

## 13.4 反思：瓶頸在哪裡

把這一章倒著讀一遍：一條 1989 年的公式，到今天仍是多國
作業系統的引擎——模型的進步其實不大。真正費力的是後面
每一步：資料的即時品質（Kaikōura 的教訓）、制度的權威性
（義大利算得出機率卻不對大眾公開）、溝通的科學（同一個 5%
可以救人也可以被告）。**從第三步起，瓶頸就不再是統計。**
對想投入這個領域的你，這是提醒也是機會：台灣的短期預報
技術零件已經齊了，缺的每一塊都不在模型裡。

不過，在瓶頸回到統計的地方，還有一件事能實質提升預報品質：
把不同的模型**組合**起來。短期的 ETAS、中期的 EEPAS、長期
的 PPE，各自抓住不同時間尺度的可預報性——
{doc}`下一章 <14_ensembles>`我們看怎麼把三個臭皮匠湊成一個
諸葛亮，以及一個著名的十年實驗如何給組合模型上了殘酷的一課。